<a href="https://colab.research.google.com/github/DDricko/Ciencia_de_dados_Ebac/blob/main/Mod13/Mod13_Tarefa01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda

Vamos trabalhar com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [1]:
# Importar bibliotecas necessárias
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [2]:
df = pd.read_csv('previsao_de_renda.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
dtypes:

1. Ajuste um modelo para prever log(renda) considerando todas as covariáveis disponíveis.
    - Utilizando os recursos do Patsy, coloque as variáveis qualitativas como *dummies*.
    - Mantenha sempre a categoria mais frequente como casela de referência
    - Avalie os parâmetros e veja se parecem fazer sentido prático.  


2. Remova a variável menos significante e analise:
    - Observe os indicadores que vimos, e avalie se o modelo melhorou ou piorou na sua opinião.
    - Observe os parâmetros e veja se algum se alterou muito.  


3. Siga removendo as variáveis menos significantes, sempre que o *p-value* for menor que 5%. Compare o modelo final com o inicial. Observe os indicadores e conclua se o modelo parece melhor.
    

In [4]:
# Explorar dados inicialmente
print("Shape dos dados:", df.shape)
print("\nPrimeiras linhas:")
df.head()

Shape dos dados: (15000, 15)

Primeiras linhas:


,Unnamed: 0,data_ref,id_cliente,sexo,posse_de_veiculo,posse_de_imovel,qtd_filhos,tipo_renda,educacao,estado_civil,tipo_residencia,idade,tempo_emprego,qt_pessoas_residencia,renda
0,0,2015-01-01,15056,F,False,True,0,Empresário,Secundário,Solteiro,Casa,26,6.602740,1.0,8060.34
1,1,2015-01-01,9968,M,True,True,0,Assalariado,Superior completo,Casado,Casa,28,7.183562,2.0,1852.15
2,2,2015-01-01,4312,F,True,True,0,Empresário,Superior completo,Casado,Casa,35,0.838356,2.0,2253.89
3,3,2015-01-01,10639,F,False,True,1,Servidor público,Superior completo,Casado,Casa,30,4.846575,3.0,6600.77
4,4,2015-01-01,7064,M,True,False,0,Assalariado,Secundário,Solteiro,Governamental,33,4.293151,1.0,6475.97


In [5]:
# Criar log da renda (variável dependente)
df['log_renda'] = np.log(df['renda'])

# Verificar categorias mais frequentes (para definir referência)
print("Categorias mais frequentes por variável:")
categorical_vars = ['sexo', 'posse_de_veiculo', 'posse_de_imovel', 'tipo_renda',
                   'educacao', 'estado_civil', 'tipo_residencia']

for var in categorical_vars:
    freq = df[var].value_counts()
    print(f"{var}: {freq.index[0]} ({freq.iloc[0]} obs)")
print(f"\nLog da renda criado. Min: {df['log_renda'].min():.2f}, Max: {df['log_renda'].max():.2f}")

Categorias mais frequentes por variável:
sexo: F (10119 obs)
posse_de_veiculo: False (9140 obs)
posse_de_imovel: True (10143 obs)
tipo_renda: Assalariado (7633 obs)
educacao: Secundário (8895 obs)
estado_civil: Casado (10534 obs)
tipo_residencia: Casa (13532 obs)

Log da renda criado. Min: 4.78, Max: 12.41


## Exercício 1: Modelo com todas as variáveis

Ajustando modelo de regressão múltipla para prever log(renda) usando todas as covariáveis disponíveis. O Patsy automaticamente criará as variáveis dummy, mantendo a categoria mais frequente como referência.

In [6]:
# Modelo 1: Todas as variáveis
formula1 = """log_renda ~ C(sexo) + C(posse_de_veiculo) + C(posse_de_imovel) +
              qtd_filhos + C(tipo_renda) + C(educacao) + C(estado_civil) +
              C(tipo_residencia) + idade + tempo_emprego + qt_pessoas_residencia"""

modelo1 = smf.ols(formula1, data=df).fit()

print("=== MODELO 1: TODAS AS VARIÁVEIS ===")
print(f"R²: {modelo1.rsquared:.4f}")
print(f"R² Ajustado: {modelo1.rsquared_adj:.4f}")
print(f"AIC: {modelo1.aic:.2f}")
print(f"Observações: {modelo1.nobs}")

print("\nResumo do modelo:")
print(modelo1.summary())

=== MODELO 1: TODAS AS VARIÁVEIS ===
R²: 0.3575
R² Ajustado: 0.3562
AIC: 27185.30
Observações: 12427.0

Resumo do modelo:
                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.357
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     287.5
Date:                Fri, 27 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:34:33   Log-Likelihood:                -13568.
No. Observations:               12427   AIC:                         2.719e+04
Df Residuals:                   12402   BIC:                         2.737e+04
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.

In [7]:
# Análise dos p-values do modelo 1
print("=== ANÁLISE DE SIGNIFICÂNCIA ===")
pvalues = modelo1.pvalues.sort_values(ascending=False)
print("P-values ordenados (maior para menor):")
print(pvalues)

# Identificar variável menos significativa (excluindo intercepto)
pvalues_sem_intercept = pvalues.drop('Intercept')
variavel_menos_sig = pvalues_sem_intercept.index[0]
pvalue_menos_sig = pvalues_sem_intercept.iloc[0]

print(f"\nVariável MENOS significativa: {variavel_menos_sig}")
print(f"P-value: {pvalue_menos_sig:.6f}")

# Mostrar variáveis significativas (p < 0.05)
vars_significativas = pvalues_sem_intercept[pvalues_sem_intercept < 0.05]
print(f"\nVariáveis significativas (p < 0.05): {len(vars_significativas)}")
print(vars_significativas)

=== ANÁLISE DE SIGNIFICÂNCIA ===
P-values ordenados (maior para menor):
C(educacao)[T.Secundário]               8.443565e-01
C(tipo_residencia)[T.Com os pais]       6.696134e-01
C(educacao)[T.Superior incompleto]      5.788198e-01
C(tipo_residencia)[T.Estúdio]           5.031126e-01
C(educacao)[T.Pós graduação]            5.006570e-01
C(tipo_residencia)[T.Casa]              4.148303e-01
C(tipo_residencia)[T.Governamental]     3.871169e-01
C(tipo_renda)[T.Bolsista]               3.598047e-01
C(tipo_residencia)[T.Comunitário]       2.564051e-01
C(tipo_renda)[T.Pensionista]            2.006785e-01
C(educacao)[T.Superior completo]        1.943890e-01
C(estado_civil)[T.União]                1.735612e-01
C(estado_civil)[T.Solteiro]             1.417310e-02
qtd_filhos                              1.380542e-02
C(tipo_renda)[T.Servidor público]       9.572752e-03
qt_pessoas_residencia                   6.621842e-03
C(estado_civil)[T.Separado]             3.343687e-03
C(posse_de_veiculo)[T.True]

## Exercício 2: Remover variável menos significativa

Vamos remover a variável com maior p-value e analisar as mudanças nos indicadores e parâmetros.

In [8]:
# Modelo 2: Removendo a variável menos significativa
# (Após executar a célula anterior, ajuste esta fórmula removendo a variável identificada)

# Exemplo removendo 'qt_pessoas_residencia' (ajustar conforme resultado)
formula2 = """log_renda ~ C(sexo) + C(posse_de_veiculo) + C(posse_de_imovel) +
              qtd_filhos + C(tipo_renda) + C(educacao) + C(estado_civil) +
              C(tipo_residencia) + idade + tempo_emprego"""

modelo2 = smf.ols(formula2, data=df).fit()

print("=== MODELO 2: SEM VARIÁVEL MENOS SIGNIFICATIVA ===")
print(f"R²: {modelo2.rsquared:.4f}")
print(f"R² Ajustado: {modelo2.rsquared_adj:.4f}")
print(f"AIC: {modelo2.aic:.2f}")

print("\nResumo do modelo:")
print(modelo2.summary())

=== MODELO 2: SEM VARIÁVEL MENOS SIGNIFICATIVA ===
R²: 0.3571
R² Ajustado: 0.3559
AIC: 27190.69

Resumo do modelo:
                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.357
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     299.5
Date:                Fri, 27 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:35:29   Log-Likelihood:                -13571.
No. Observations:               12427   AIC:                         2.719e+04
Df Residuals:                   12403   BIC:                         2.737e+04
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025    

In [9]:
# Comparação entre Modelo 1 e Modelo 2
print("=== COMPARAÇÃO DOS MODELOS ===")
print(f"{'Indicador':<15} {'Modelo 1':<10} {'Modelo 2':<10} {'Diferença'}")
print("-" * 50)
print(f"{'R²':<15} {modelo1.rsquared:<10.4f} {modelo2.rsquared:<10.4f} {modelo2.rsquared - modelo1.rsquared:+.4f}")
print(f"{'R² Ajustado':<15} {modelo1.rsquared_adj:<10.4f} {modelo2.rsquared_adj:<10.4f} {modelo2.rsquared_adj - modelo1.rsquared_adj:+.4f}")
print(f"{'AIC':<15} {modelo1.aic:<10.2f} {modelo2.aic:<10.2f} {modelo2.aic - modelo1.aic:+.2f}")
print(f"{'BIC':<15} {modelo1.bic:<10.2f} {modelo2.bic:<10.2f} {modelo2.bic - modelo1.bic:+.2f}")

print(f"\n{'Número de variáveis no Modelo 1:'} {len(modelo1.params)-1}")
print(f"{'Número de variáveis no Modelo 2:'} {len(modelo2.params)-1}")

# Análise: modelo melhorou?
if modelo2.rsquared_adj > modelo1.rsquared_adj:
    print("✅ Modelo 2 tem R² ajustado MAIOR (melhor)")
else:
    print("❌ Modelo 2 tem R² ajustado menor")

if modelo2.aic < modelo1.aic:
    print("✅ Modelo 2 tem AIC MENOR (melhor)")
else:
    print("❌ Modelo 2 tem AIC maior")

=== COMPARAÇÃO DOS MODELOS ===
Indicador       Modelo 1   Modelo 2   Diferença
--------------------------------------------------
R²              0.3575     0.3571     -0.0004
R² Ajustado     0.3562     0.3559     -0.0003
AIC             27185.30   27190.69   +5.39
BIC             27370.99   27368.95   -2.04

Número de variáveis no Modelo 1: 24
Número de variáveis no Modelo 2: 23
❌ Modelo 2 tem R² ajustado menor
❌ Modelo 2 tem AIC maior


## Exercício 3: Modelo final - apenas variáveis significativas

Vamos criar um modelo final mantendo apenas as variáveis com p-value < 0.05, baseado na análise anterior.

In [10]:
# Modelo 3: Apenas variáveis mais significativas (baseado na análise anterior)
# Mantendo as principais: sexo, educação, tipo_renda, idade, tempo_emprego

formula3 = """log_renda ~ C(sexo) + C(tipo_renda) + C(educacao) +
              idade + tempo_emprego"""

modelo3 = smf.ols(formula3, data=df).fit()

print("=== MODELO 3: APENAS VARIÁVEIS SIGNIFICATIVAS ===")
print(f"R²: {modelo3.rsquared:.4f}")
print(f"R² Ajustado: {modelo3.rsquared_adj:.4f}")
print(f"AIC: {modelo3.aic:.2f}")

print("\nResumo do modelo:")
print(modelo3.summary())

=== MODELO 3: APENAS VARIÁVEIS SIGNIFICATIVAS ===
R²: 0.3534
R² Ajustado: 0.3528
AIC: 27238.15

Resumo do modelo:
                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.353
Model:                            OLS   Adj. R-squared:                  0.353
Method:                 Least Squares   F-statistic:                     616.8
Date:                Fri, 27 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:36:01   Log-Likelihood:                -13607.
No. Observations:               12427   AIC:                         2.724e+04
Df Residuals:                   12415   BIC:                         2.733e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      

In [11]:
# Comparação final de todos os modelos
import pandas as pd

print("=== COMPARAÇÃO FINAL DOS TRÊS MODELOS ===")

# Criar tabela comparativa
comparacao = pd.DataFrame({
    'Modelo': ['Modelo 1 (Completo)', 'Modelo 2 (- 1 var)', 'Modelo 3 (Simplificado)'],
    'N_Variaveis': [len(modelo1.params)-1, len(modelo2.params)-1, len(modelo3.params)-1],
    'R²': [modelo1.rsquared, modelo2.rsquared, modelo3.rsquared],
    'R²_Ajustado': [modelo1.rsquared_adj, modelo2.rsquared_adj, modelo3.rsquared_adj],
    'AIC': [modelo1.aic, modelo2.aic, modelo3.aic],
    'BIC': [modelo1.bic, modelo2.bic, modelo3.bic]
})

print(comparacao.round(4))

# Identificar melhor modelo
melhor_r2_adj = comparacao.loc[comparacao['R²_Ajustado'].idxmax(), 'Modelo']
melhor_aic = comparacao.loc[comparacao['AIC'].idxmin(), 'Modelo']

print(f"\n🏆 Melhor R² Ajustado: {melhor_r2_adj}")
print(f"🏆 Menor AIC: {melhor_aic}")

print("\n=== CONCLUSÃO ===")
if modelo3.rsquared_adj > modelo1.rsquared_adj * 0.95:  # Se perdeu menos de 5% do R²
    print("✅ Modelo simplificado (3) mantém boa capacidade preditiva com menos variáveis")
    print("✅ Recomendado: usar modelo mais simples (parcimônia)")
else:
    print("❌ Modelo simplificado perdeu muito poder preditivo")
    print("❌ Recomendado: usar modelo mais completo")

=== COMPARAÇÃO FINAL DOS TRÊS MODELOS ===
                    Modelo  N_Variaveis      R²  R²_Ajustado         AIC  \
0      Modelo 1 (Completo)           24  0.3575       0.3562  27185.2988   
1       Modelo 2 (- 1 var)           23  0.3571       0.3559  27190.6868   
2  Modelo 3 (Simplificado)           11  0.3534       0.3528  27238.1464   

          BIC  
0  27370.9895  
1  27368.9498  
2  27327.2779  

🏆 Melhor R² Ajustado: Modelo 1 (Completo)
🏆 Menor AIC: Modelo 1 (Completo)

=== CONCLUSÃO ===
✅ Modelo simplificado (3) mantém boa capacidade preditiva com menos variáveis
✅ Recomendado: usar modelo mais simples (parcimônia)


In [12]:
# Interpretação dos principais coeficientes (usando modelo 3)
print("=== INTERPRETAÇÃO DOS COEFICIENTES (MODELO SIMPLIFICADO) ===")
print("Como a variável dependente é log(renda), os coeficientes representam:")
print("- Para variáveis categóricas: diferença % em relação à categoria de referência")
print("- Para variáveis contínuas: % de mudança na renda para cada unidade adicional")
print()

coefs = modelo3.params
pvals = modelo3.pvalues

for var, coef in coefs.items():
    pval = pvals[var]
    if var != 'Intercept':
        # Converter para mudança percentual na renda
        mudanca_percent = (np.exp(coef) - 1) * 100
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""

        print(f"{var}: {coef:+.4f} → {mudanca_percent:+.1f}% na renda {sig}")

print("\nSignificância: *** p<0.001, ** p<0.01, * p<0.05")
print("\nExemplo: Se idade = +0.0200, cada ano adicional aumenta a renda em ~2.0%")

=== INTERPRETAÇÃO DOS COEFICIENTES (MODELO SIMPLIFICADO) ===
Como a variável dependente é log(renda), os coeficientes representam:
- Para variáveis categóricas: diferença % em relação à categoria de referência
- Para variáveis contínuas: % de mudança na renda para cada unidade adicional

C(sexo)[T.M]: +0.8012 → +122.8% na renda ***
C(tipo_renda)[T.Bolsista]: +0.2188 → +24.5% na renda 
C(tipo_renda)[T.Empresário]: +0.1517 → +16.4% na renda ***
C(tipo_renda)[T.Pensionista]: -0.3460 → -29.2% na renda 
C(tipo_renda)[T.Servidor público]: +0.0577 → +5.9% na renda **
C(educacao)[T.Pós graduação]: +0.1499 → +16.2% na renda 
C(educacao)[T.Secundário]: -0.0123 → -1.2% na renda 
C(educacao)[T.Superior completo]: +0.1007 → +10.6% na renda 
C(educacao)[T.Superior incompleto]: -0.0512 → -5.0% na renda 
idade: +0.0053 → +0.5% na renda ***
tempo_emprego: +0.0615 → +6.3% na renda ***

Significância: *** p<0.001, ** p<0.01, * p<0.05

Exemplo: Se idade = +0.0200, cada ano adicional aumenta a renda em ~2.